# CineScore Phase 4: The LLM Narrative Engine (V11 Groq + Cohere Edition)
Integrating the ultimate Enterprise-Grade failover: A dual-cloud API router that dynamically load-balances across Groq LPUs natively, and seamlessly catches any total-cluster failures by dynamically routing the payload to Cohere's unrestricted `Command-R` Enterprise endpoint.

In [ ]:
%pip install -q -U groq cohere tqdm python-dotenv


In [ ]:
import pandas as pd
import numpy as np
import time
import json
import os
import shutil
from tqdm import tqdm

# 🛑 PERMANENT STORAGE PROTOCOL & SMART UPLOADER
if os.path.exists('/content'):
    print("⏳ Initializing Permanent Storage Architecture...")
    try:
        from google.colab import drive, files
    except ImportError:
        pass
        
    drive.mount('/content/drive')
    print("✅ Drive Mounted Protocol Secure.")
    
    DRIVE_DIR = "/content/drive/MyDrive/CineScore"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    
    CHECKPOINT_PATH = f"{DRIVE_DIR}/v7_nlp_analytical_df_CHECKPOINT.csv"
    OUTPUT_PATH = f"{DRIVE_DIR}/v7_nlp_analytical_df.csv"
    
    v6_drive_path = f"{DRIVE_DIR}/v6_master_analytical_df.csv"
    
    if os.path.exists(v6_drive_path):
        print(f"✅ Found `v6_master_analytical_df.csv` safely backed up on your Google Drive!")
        INPUT_PATH = v6_drive_path
    else:
        print("\n⚠️ Baseline dataset missing from Google Drive. Please click 'Choose Files' below to upload your `v6` dataset:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        
        print("💾 Synchronizing baseline directly onto your Google Drive for permanent safety...")
        shutil.copy(filename, v6_drive_path)
        INPUT_PATH = v6_drive_path
        
else:
    print("Detected Local Environment")
    INPUT_PATH = "../Data/Processed_Dataset/v6_master_analytical_df.csv"
    CHECKPOINT_PATH = "../Data/Processed_Dataset/v7_nlp_analytical_df_CHECKPOINT.csv"
    OUTPUT_PATH = "../Data/Processed_Dataset/v7_nlp_analytical_df.csv"


### Step 1: Enterprise Auto-Recovery Protocol

In [ ]:
def ensure_overview_exists(dataframe):
    if 'overview' not in dataframe.columns:
        print("⚠️ Overview missing! Ensure you inputted an uncompressed master DF.")
    return dataframe

start_index = 0

if os.path.exists(CHECKPOINT_PATH):
    print(f"\n💾 RESUME PROTOCOL ENGAGED: Found isolated Checkpoint File located securely on Drive.")
    df = pd.read_csv(CHECKPOINT_PATH)
    
    unprocessed_mask = df['four_quadrant_appeal'].isna()
    if unprocessed_mask.any():
        start_index = unprocessed_mask.idxmax()
        print(f"Resuming Groq+Cohere API extraction natively forward from exact Row Index: {start_index}...")
    else:
        print("All rows appear to be already processed in the checkpoint!")
        start_index = df.shape[0]
else:
    print(f"\nLoading Fresh ML Dataset: {INPUT_PATH}")
    try:
        df = pd.read_csv(INPUT_PATH)
    except FileNotFoundError:
        raise FileNotFoundError(f"{INPUT_PATH} not found. Please double check pathing or upload location.")
        
    df = ensure_overview_exists(df)

    initial_len = df.shape[0]
    df = df[df['overview'].notna()]
    df = df[df['overview'].astype(str).str.split().str.len() >= 20].copy()
    df = df.reset_index(drop=True)
    print(f"Retained {df.shape[0]} valid narrative rows.")
    
    df['four_quadrant_appeal'] = np.nan
    df['high_concept_marketability'] = np.nan


### Step 2: Groq + Cohere Dual-Cloud Engine

In [ ]:
from groq import Groq
import cohere

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    COHERE_API_KEY = userdata.get('COHERE_API_KEY')
    print("🔑 Secure Handshake: Locked firmly onto Colab Native Secrets (Groq + Cohere).")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    GROQ_API_KEY = os.getenv('GROQ_API_KEY')
    COHERE_API_KEY = os.getenv('COHERE_API_KEY')
    print("💻 Colab bypassed. Locked firmly onto local .env credential.")

SYSTEM_PROMPT = """
You are a Senior Hollywood Studio Executive calculating risk matrices for greenlighting scripts.
Evaluate the user's plot overview strictly on two commercial metrics on a 1.0 to 10.0 scale.

Metric 1 - "four_quadrant_appeal": Does this appeal uniformly to all demographics (kids, teens, adults, seniors), or is it deeply niche, complex, or R-rated? (Higher score = broader audience).
Metric 2 - "high_concept_marketability": Is the plot highly original, easy to pitch in 5 words, and extremely easy to sell on a billboard? (Higher score = instantly understandable hook).

OUTPUT RULES:
1. Return ONLY a valid JSON object.
2. Absolutely no conversational filler.
3. Do NOT wrap the JSON in markdown code blocks (do not use ```json).
4. Start your response immediately with the { character.
5. Values must be floats with one decimal place.

EXPECTED SCHEMA:
{
  "four_quadrant_appeal": 8.5,
  "high_concept_marketability": 7.0
}
"""

client_groq = Groq(api_key=GROQ_API_KEY)
client_cohere = cohere.Client(api_key=COHERE_API_KEY)

def extract_semantic_variables(overview):
    user_prompt = f'Plot Overview: "{overview}"'
    
    for cycle_attempt in range(5):
        # TIER 1: The Groq Baseline
        try:
            chat_completion = client_groq.chat.completions.create(
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                model="mixtral-8x7b-32768",
                temperature=0.0,
                response_format={"type": "json_object"}
            )
            
            result = json.loads(chat_completion.choices[0].message.content)
            return float(np.clip(result.get('four_quadrant_appeal', 5.0), 1.0, 10.0)), float(np.clip(result.get('high_concept_marketability', 5.0), 1.0, 10.0))
            
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                pass 
            else:
                pass
        
        # TIER 2: Cohere Enterprise Fallback! (If Groq congests, Cohere saves the row instantly)
        try:
            cohere_response = client_cohere.chat(
                model="command-r",
                message=f"{SYSTEM_PROMPT}\n\n{user_prompt}",
                temperature=0.0
            )
            # Sanitizing markdown drift if it occurs
            clean_json = cohere_response.text.replace("```json", "").replace("```", "").strip()
            result = json.loads(clean_json)
            return float(np.clip(result.get('four_quadrant_appeal', 5.0), 1.0, 10.0)), float(np.clip(result.get('high_concept_marketability', 5.0), 1.0, 10.0))
        except Exception as e:
            if "429" in str(e) or "limit" in str(e).lower():
                pass
            else:
                print(f"\n⚠️ Cohere Parse Error: {e}")
                pass

        sleep_time = 15 * (2 ** cycle_attempt)
        tqdm.write(f"\n⏳ [DUAL-CLOUD CONGESTED] Hitting Rate Limit. Dropping back for {sleep_time}s...")
        time.sleep(sleep_time)
                
    return "QUOTA_DEAD", "QUOTA_DEAD"


### Step 3: Massive Iterative Ping Execution

In [ ]:
print("Initiating Massive LLM Checkpoint Extraction...")

pbar = tqdm(total=df.shape[0], initial=start_index, desc="Groq+Cohere Phase")

for index in range(start_index, df.shape[0]):
    plot_text = df.at[index, 'overview']
    
    q4, hc = extract_semantic_variables(plot_text)
    
    if q4 == "QUOTA_DEAD":
        tqdm.write(f"\n🚨 [HARD STOP] Model Server Exhaustion. Progress safely frozen at Row {index}.")
        df.to_csv(CHECKPOINT_PATH, index=False)
        break
    
    df.at[index, 'four_quadrant_appeal'] = q4
    df.at[index, 'high_concept_marketability'] = hc

    if (index + 1) % 50 == 0:
        df.to_csv(CHECKPOINT_PATH, index=False)
        tqdm.write(f"\n💾 [Checkpoint] State safely cemented to Google Drive at Row {index + 1}.")

    # Sleeping 5.0s mechanically locks the loop to precisely 10 Requests Per Minute.
    time.sleep(5.0)
    pbar.update(1)
    
pbar.close()


### Step 4: The Clean NLP Export

In [ ]:
if 'four_quadrant_appeal' in df.columns and df['four_quadrant_appeal'].isna().sum() == 0:
    if 'overview' in df.columns:
        df = df.drop(columns=['overview'])

    df.to_csv(OUTPUT_PATH, index=False)
    if os.path.exists(CHECKPOINT_PATH):
        os.remove(CHECKPOINT_PATH)
    print(f"\n🔥 NARRATIVE ENGINE V11 COMPLETE 🔥")
    print(f"V7 Artifact finalized and secured statically to: {OUTPUT_PATH}")
else:
    print("\n⚠️ Pipeline halted early via Quota block. Checkpoint is safely active. Come back later.")
